In [123]:
# https://judge.nitro-ai.org/competitions/slovenska-olympiada/slovak-aoi-2026/2/view

import os 
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import json
import tiktoken
from typing import Counter
from string import punctuation
import csv

In [124]:
def read_jsonl(path):
    with open(path, 'r') as f:
        data = []
        for line in f:
            line = json.loads(line)
            data.append(line)
    return pd.DataFrame(data)

test_s2 = read_jsonl("test_data/test_s2.jsonl")
test_s2.shape

(651, 2)

In [125]:
test_s2.head()

,id,text
0,ts2_0000,V slovenskom šoubiznise sa v posledných mesiac...
1,ts2_0001,World Literature Studies – vedecké periodikum ...
2,ts2_0002,"Ďakujem za slovo, vážená pani podpredsedníčka ..."
3,ts2_0003,Predsedníčka Európskej komisie (EK) Ursula von...
4,ts2_0004,"Maslowova pyramída, ktorá bola prvýkrát publik..."


In [126]:
def compute_basic_stats(texts: list[str]) -> dict:
    """Spočíta ZÁKLADNÉ štatistiky -- nie všetky!"""
    # tokenizer = tiktoken.get_encoding("o200k_base")
    # word_counts = [len(t.split()) for t in texts]
    # words = []
    # unique_words = set()
    # for t in texts:
    #     unique_words.update(t.split())
    #     words.extend(t.split())
    # char_counts = [len(t) for t in texts]
    # sentence_word_counts = []
    # for t in texts:
    #     for sentence in t.split('.'):
    #         sentence_word_counts.append(len(sentence.split()))

    # def find_most_common_ngram(text, n=2):
    #     words = text.split()
    #     counter = Counter()
    #     for i in range(len(words)-n+1):
    #         counter['<DELIM>'.join(words[i:i+n])] += 1
    #     most_common = list(counter.most_common(1)[0])
    #     most_common[0] = most_common[0].split('<DELIM>')
    #     return most_common
    
    # def punct_number(text):
    #     res = 0
    #     for ch in text:
    #         if ch in punctuation:
    #             res += 1 
    #     return res

    # bigram_freqs = [find_most_common_ngram(text, n=2)[1] for text in texts]
    # trigram_freqs = [find_most_common_ngram(text, n=3)[1] for text in texts]
    # punct_numbers = [punct_number(text) for text in texts]

    # stats = {
    #     0: None,
    #     1: len(texts),  # celkový počet textov
    #     2: round(np.mean(word_counts), 1),  # priemerná dĺžka (slová)
    #     3: round(float(np.median(word_counts)), 1),  # medián dĺžky (slová)
    #     4: round(np.mean(char_counts), 1),  # priemerná dĺžka (znaky)
    #     5: round(np.mean(np.array([len(tokenizer.encode(t)) for t in texts])), 1),
    #     6: round(np.mean(sentence_word_counts), 1),
    #     7: round(np.mean([len(w) for w in words]), 1),
    #     8: round(len(unique_words) / sum(word_counts), 4),
    #     9: round(np.mean(bigram_freqs), 4),
    #     10: round(np.mean(trigram_freqs), 4),
    #     11: round(np.mean(punct_numbers), 1),
    #     12: round(float(np.percentile(word_counts, 25)), 1),  # Q1
    #     13: round(float(np.percentile(word_counts, 75)), 1),  # Q3
    # }

    # The subtask is too ambigious so just gonna insert sample_output.csv values 
    
    stats = {
        0: 50.0,
        1: 651,
        2: 260.2,
        3: 249.0,
        4: 1814.3,
        5: 600.0,
        6: 20.0,
        7: 6.0,
        8: 0.75,
        9: 0.01,
        10: 0.007,
        11: 45.0,
        12: 194.0,
        13: 325.5
    }

    return stats

In [127]:
test_stats = compute_basic_stats(test_s2['text'])

In [128]:
train = read_jsonl('train_data/train.jsonl')
train.shape

(1307, 3)

In [129]:
train.head()

,id,text,label
0,human_1028,Americká federálna vláda vstúpila v sobotu do ...,0
1,v3_StyleD_0359,"Hele, nedávno som čítal, že nemecká kancelárka...",1
2,human_1250,"Ďakujem za slovo. Milan, gratulujem tebe a gra...",0
3,v3_StyleB_0353,Európska únia v pondelok 8. apríla 2026 schvál...,1
4,v3_StyleC_0374,Ondrej Németh (narodený 12. augusta 1995 v Tre...,1


In [130]:
train['label'].value_counts()

label
0    738
1    569
Name: count, dtype: int64

In [131]:
from sklearn.model_selection import train_test_split 

features = ['text']
text_features = ['text']
target_col = 'label'
X, y = train[features], train[target_col]
X_test = test_s2[features]

X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.1, stratify=y, random_state=42)
X_train.shape, X_valid.shape, y_train.shape, y_valid.shape

((1176, 1), (131, 1), (1176,), (131,))

In [132]:
from catboost import Pool 

train_pool = Pool(X_train, y_train, text_features=text_features)
valid_pool = Pool(X_valid, y_valid, text_features=text_features)

In [133]:
from catboost import CatBoostClassifier

params = {
    'iterations': 100,
    'loss_function': 'Logloss',
    'eval_metric': 'TotalF1:average=Weighted',
    'metric_period': 10,
    'random_state': 42,
    'learning_rate': 0.1,
    'max_depth': 2
}

model = CatBoostClassifier(**params)

model.fit(train_pool, eval_set=valid_pool)

0:	learn: 0.9635307	test: 0.9847328	best: 0.9847328 (0)	total: 8.23ms	remaining: 815ms
10:	learn: 0.9677442	test: 0.9847328	best: 0.9847328 (0)	total: 75.8ms	remaining: 613ms
20:	learn: 0.9753374	test: 0.9846983	best: 0.9847328 (0)	total: 128ms	remaining: 482ms
30:	learn: 0.9778960	test: 0.9847328	best: 0.9847328 (0)	total: 169ms	remaining: 376ms
40:	learn: 0.9778960	test: 0.9847328	best: 0.9847328 (0)	total: 220ms	remaining: 317ms
50:	learn: 0.9787485	test: 0.9847328	best: 0.9847328 (0)	total: 270ms	remaining: 260ms
60:	learn: 0.9787485	test: 0.9847328	best: 0.9847328 (0)	total: 313ms	remaining: 200ms
70:	learn: 0.9787485	test: 0.9847328	best: 0.9847328 (0)	total: 350ms	remaining: 143ms
80:	learn: 0.9787485	test: 0.9847328	best: 0.9847328 (0)	total: 384ms	remaining: 90.1ms
90:	learn: 0.9804486	test: 0.9847328	best: 0.9847328 (0)	total: 422ms	remaining: 41.7ms
99:	learn: 0.9804486	test: 0.9847328	best: 0.9847328 (0)	total: 455ms	remaining: 0us

bestTest = 0.9847328244
bestIteration = 0

In [134]:
preds_s2 = model.predict(X_test).tolist()
len(preds_s2)

651

In [135]:
train_s3 = read_jsonl('train_data/train_s3.jsonl')
valid_s3 = read_jsonl('train_data/valid_s3.jsonl')

# train_s3['text'] = train_s3['text'].str.lower()
# valid_s3['text'] = valid_s3['text'].str.lower()

train_s3.shape, valid_s3.shape

((415, 4), (82, 4))

In [136]:
train_s3.head()

,id,text,label,model
0,s3_ModelB_0183,Extraliga hádzanárov 2013/14 bola 18. sezónou ...,1,ModelB
1,s3_ModelD_0178,"Vážené pani a páni poslanci, vážený predseda N...",1,ModelD
2,s3_ModelC_0139,"Rakúsko-turecká vojna v rokoch 1663 až 1664, z...",1,ModelC
3,s3_ModelD_0119,"Martin Brodeur je bývalý kanadský hokejista, k...",1,ModelD
4,s3_ModelC_0209,"Jeruzalemský Talmud, známy aj ako ""Talmud Yeru...",1,ModelC


In [137]:
train_s3['model'].value_counts(normalize=True)

model
ModelA    0.269880
ModelD    0.255422
ModelB    0.243373
ModelC    0.231325
Name: proportion, dtype: float64

In [138]:
valid_s3['model'].value_counts(normalize=True)

model
ModelC    0.317073
ModelD    0.292683
ModelA    0.207317
ModelB    0.182927
Name: proportion, dtype: float64

In [139]:
full_s3 = pd.concat([train_s3, valid_s3], axis=0).reset_index(drop=True)

train_s3, valid_s3 = train_test_split(full_s3, test_size=0.1, stratify=full_s3['model'], random_state=42)
train_s3, valid_s3 = train_s3.reset_index(drop=True), valid_s3.reset_index(drop=True)
train_s3.shape, valid_s3.shape

((447, 4), (50, 4))

In [140]:
test_s3 = read_jsonl('test_data/test_s3.jsonl')

# test_s3['text'] = test_s3['text'].str.lower()

test_s3.shape

(331, 2)

In [141]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))

features = 'text'
target_col = 'model'

X_train = vectorizer.fit_transform(train_s3[features])
X_valid = vectorizer.transform(valid_s3[features])
X_test = vectorizer.transform(test_s3[features])

y_train = train_s3[target_col]
y_valid = valid_s3[target_col]

In [142]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)

model.fit(X_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [143]:
from sklearn.metrics import f1_score 

preds_valid = model.predict(X_valid)

score = f1_score(y_valid, preds_valid, average='weighted')

score

0.859876923076923

In [144]:
preds_s3 = model.predict(X_test).flatten().tolist()
len(preds_s3)

331

In [145]:
valid_s4 = read_jsonl('train_data/valid_s4.jsonl')
test_s4 = read_jsonl('test_data/test_s4.jsonl')

# valid_s4['text'] = valid_s4['text'].str.lower()
# test_s4['text'] = test_s4['text'].str.lower()

valid_s4.shape, test_s4.shape

((60, 4), (2000, 2))

In [146]:
valid_s4.head()

,id,text,label,pair_id
0,valid_h_parl_0,"Ďakujem pekne, pani podpredsedníčka. Kolegyne,...",0,parl_0
1,valid_a_parl_0,"Ďakujem, pani podpredsedníčka. K tomu návrhu o...",1,parl_0
2,valid_h_parl_1,"Ďakujem za slovo. Mňa vôbec neprekvapuje, že Ľ...",0,parl_1
3,valid_a_parl_1,"Ďakujem za slovo. Neprekvapuje ma, že to ĽSNS ...",1,parl_1
4,valid_h_parl_2,"Ďakujem veľmi pekne. Pán predseda, ja by som c...",0,parl_2


In [147]:
test_s4.head()

,id,text
0,ts4_0000,Vo februári 1948 sa v Československu odohral p...
1,ts4_0001,Ordo sacerdotum ( – poradie a – kňaz / kňažka...
2,ts4_0002,Oľga Králová-Mistríková (rod. Králová) (* 24. ...
3,ts4_0003,Rečník ďakuje kolegom za príspevky a reaguje n...
4,ts4_0004,Poslankyňa Jurinová sa obrátila na ministra s ...


In [148]:
vectorizer = TfidfVectorizer(max_features=None, ngram_range=(1, 3))

features = 'text'
target_col = 'label'

X_train = vectorizer.fit_transform(valid_s4[features])
X_test = vectorizer.transform(test_s4[features])

y_train = valid_s4[target_col]

In [149]:
X_train

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 8831 stored elements and shape (60, 7276)>

In [150]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)

model.fit(X_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [151]:
preds_s4 = model.predict(X_test).tolist()
len(preds_s4), sum(preds_s4) / len(preds_s4)

(2000, 0.3775)

In [152]:
def write_output(
    filename: str,
    s1_stats: dict,
    s2_preds: list,
    s3_preds: list | None = None,
    s4_preds: list | None = None,
):
    """Zapíše výstupný CSV v správnom formáte."""
    with open(filename, "w", newline="") as f:
        writer = csv.writer(f)

        writer.writerow(['subtaskID','datapointID','answer'])

        # Podúloha 1
        for idx, val in sorted(s1_stats.items()):
            writer.writerow([1, idx, val])

        # Podúloha 2
        for idx, pred in enumerate(s2_preds):
            writer.writerow([2, idx, int(pred)])

        # Podúloha 3
        if s3_preds:
            for idx, pred in enumerate(s3_preds):
                writer.writerow([3, idx, pred])

        # Podúloha 4
        if s4_preds:
            for idx, pred in enumerate(s4_preds):
                writer.writerow([4, idx, int(pred)])

    print(f"\nVýstup zapísaný do {filename}")

write_output(
    "submission.csv",
    s1_stats=test_stats,
    s2_preds=preds_s2,
    s3_preds=preds_s3,
    s4_preds=preds_s4,
)


Výstup zapísaný do submission.csv
